# WiDS Global Datathon 2026 — Wildfire Threat Probability Modeling

**Purpose:** build a reproducible baseline for predicting whether a wildfire will reach within 5 km of an evacuation-zone centroid within 12, 24, 48, or 72 hours.

This notebook:
1. loads the competition files;
2. inspects and validates the data;
3. engineers a few domain-informed features when the required columns exist;
4. trains a separate probability model for each time horizon while respecting right-censoring;
5. checks validation Brier scores, creates monotonic probabilities, and writes a Kaggle submission.

**Important:** this is a sounder baseline than predicting a single time and converting it with an arbitrary exponential formula. A fire censored before a horizon has an unknown outcome at that horizon and must not be treated as a negative example.

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0. Put the files in one folder

Place these files in the same folder as this notebook (or update `DATA_DIR` below):

- `train.csv` (or `train.xlsx`)
- `test.csv` (or `test.xlsx`)
- `sample_submission.csv` (or `sample_submission.xlsx`)
- `metadata.csv` / `metadata.xlsx` (optional; filename may vary)

The code supports CSV and Excel files. If your files are named differently, edit the filenames in the configuration cell.

In [20]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

RANDOM_STATE = 42
DATA_DIR = Path(".")  # change this to your data folder if needed

TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"
SAMPLE_FILE = "sample_submission.csv"
METADATA_FILE = "metadata.csv"  # optional; change if yours has another name

HORIZONS = [12, 24, 48, 72]
ID_COL = "event_id"
TIME_COL = "time_to_hit_hours"
EVENT_COL = "event"

print("Notebook folder:", Path.cwd())

Notebook folder: /content


## 1. Load the competition files

If XGBoost and LightGBM are installed, the notebook uses them. Otherwise it falls back to scikit-learn models, so you can still run the workflow.

In [21]:
def read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path.resolve()}. Check DATA_DIR and the filename."
        )
    suffix = path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {suffix}. Use CSV or Excel.")

train = read_table(DATA_DIR / TRAIN_FILE)
test = read_table(DATA_DIR / TEST_FILE)
sample = read_table(DATA_DIR / SAMPLE_FILE)

try:
    metadata = read_table(DATA_DIR / METADATA_FILE)
except FileNotFoundError:
    metadata = None
    print("Metadata file not found under the configured name; continuing without it.")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample.shape)
print("\nTrain columns:")
print(train.columns.tolist())

Metadata file not found under the configured name; continuing without it.
Train shape: (221, 37)
Test shape: (95, 35)
Sample submission shape: (95, 5)

Train columns:
['event_id', 'num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first', 'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h', 'radial_growth_m', 'radial_growth_rate_m_per_h', 'centroid_displacement_m', 'centroid_speed_m_per_h', 'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos', 'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h', 'dist_slope_ci_0_5h', 'closing_speed_m_per_h', 'closing_speed_abs_m_per_h', 'projected_advance_m', 'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos', 'alignment_abs', 'cross_track_component', 'along_track_speed', 'event_start_hour', 'event_start_dayofweek', 'event_start_month', 'time_to_hit_hours', 'event']


## 2. Inspect the data before modeling

Check column names, data types, missing values, target ranges, and whether IDs match the sample submission. Do not skip this step: it catches many avoidable errors.

In [22]:
display(train.head())
display(test.head())
display(sample.head())

print("\nTraining data types:")
display(train.dtypes.to_frame("dtype"))

print("\nMissing values in training data (top 20):")
display(train.isna().sum().sort_values(ascending=False).head(20).to_frame("missing_count"))

required_train = {ID_COL, TIME_COL, EVENT_COL}
missing_required = required_train - set(train.columns)
if missing_required:
    raise ValueError(
        f"Training data is missing required columns: {missing_required}. "
        f"Check the exact names in train.columns."
    )

required_sample = {ID_COL, "prob_12h", "prob_24h", "prob_48h", "prob_72h"}
if not required_sample.issubset(sample.columns):
    print("Sample submission columns:", sample.columns.tolist())
    raise ValueError(f"Sample submission must contain: {sorted(required_sample)}")

print("\nEvent counts:")
display(train[EVENT_COL].value_counts(dropna=False).to_frame("count"))

print("\nTime-to-hit summary:")
display(train[TIME_COL].describe().to_frame("time_to_hit_hours"))

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0


,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,projected_advance_m,dist_accel_m_per_h2,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month
0,10662602,1,0.000000,1,2.452217,0.000000,0.00000,0.000000,1.239017,0.000000,...,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0,3,7
1,13353600,1,0.000000,1,131.669588,0.000000,0.00000,0.000000,4.887862,0.000000,...,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,22,0,8
2,13942327,1,0.000000,1,6.723104,0.000000,0.00000,0.000000,2.044216,0.000000,...,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,2,6,7
3,16112781,1,0.000000,1,285.416736,0.000000,0.00000,0.000000,5.657448,0.000000,...,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0,1,7
4,17132808,7,3.459331,0,61.098604,12.516633,0.20486,3.618224,4.128724,2.603921,...,13.54413,-22.687575,0.044572,0.15855,0.15855,-24.414806,3.920562,23,5,7


,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.5,0.5,0.5,0.5
1,13353600,0.5,0.5,0.5,0.5
2,13942327,0.5,0.5,0.5,0.5
3,16112781,0.5,0.5,0.5,0.5
4,17132808,0.5,0.5,0.5,0.5



Training data types:


,dtype
event_id,int64
num_perimeters_0_5h,int64
dt_first_last_0_5h,float64
low_temporal_resolution_0_5h,int64
area_first_ha,float64
area_growth_abs_0_5h,float64
area_growth_rel_0_5h,float64
area_growth_rate_ha_per_h,float64
log1p_area_first,float64
log1p_growth,float64


...

,missing_count
event_id,0
num_perimeters_0_5h,0
dt_first_last_0_5h,0
low_temporal_resolution_0_5h,0
area_first_ha,0
area_growth_abs_0_5h,0
area_growth_rel_0_5h,0
area_growth_rate_ha_per_h,0
log1p_area_first,0
log1p_growth,0



Event counts:


,count
event,
0,152
1,69



Time-to-hit summary:


,time_to_hit_hours
count,221.000000
mean,37.567626
std,25.902361
min,0.001220
25%,12.242322
50%,43.109830
75%,63.938706
max,66.994474


## 3. Understand the survival target

- `event = 1`: the fire reached the defined threat distance during observation; `time_to_hit_hours` is the observed time.
- `event = 0`: the observation is right-censored; the fire was not observed to hit before observation ended.
- For a horizon `H`, an observation is usable only when its status by `H` is known:
  - positive: `event == 1` and time `<= H`;
  - negative: observed hit after `H`, or censored at/after `H`;
  - unknown and excluded: censored before `H`.

This horizon-specific target construction prevents early-censored fires from being incorrectly labeled as safe at later horizons.

In [23]:
print("event values:", sorted(train[EVENT_COL].dropna().unique().tolist()))
print("Minimum time:", train[TIME_COL].min())
print("Maximum time:", train[TIME_COL].max())

if not set(train[EVENT_COL].dropna().unique()).issubset({0, 1}):
    raise ValueError("The event column should contain 0/1 values.")
if (train[TIME_COL].dropna() < 0).any():
    raise ValueError("Negative time_to_hit_hours values found; investigate before modeling.")

event values: [0, 1]
Minimum time: 0.0012202119444444
Maximum time: 66.99447413277778


## 4. Prepare features and add carefully checked features

We exclude the ID and outcome columns from predictors. The feature-engineering section only creates a feature when its source columns exist. This avoids hard-coding assumptions about every column name.

Missing numeric values are imputed using the training fold's median inside each model pipeline. Non-numeric columns are excluded in this baseline; inspect them and encode them separately if meaningful categorical features exist.

In [24]:
# Keep IDs for submission, but never use IDs as model features.
excluded = {ID_COL, TIME_COL, EVENT_COL}
candidate_features = [c for c in train.columns if c not in excluded and c in test.columns]

X_all = train[candidate_features].copy()
X_test_all = test.reindex(columns=candidate_features).copy()

# Convert candidate columns to numeric where possible; non-numeric columns are excluded here.
numeric_cols = X_all.select_dtypes(include=[np.number, "bool"]).columns.tolist()
X_all = X_all[numeric_cols].copy()
X_test_all = X_test_all.reindex(columns=numeric_cols).copy()

def add_domain_features(frame):
    frame = frame.copy()
    # Use these only if the exact source columns exist.
    distance_candidates = ["dist_min_ci_0_5h", "dist_min_ci", "dist_fir_t2_0_5h"]
    speed_candidates = ["closing_speed_abs_m_per_h", "closing_speed", "radial_growth_m"]

    dist_col = next((c for c in distance_candidates if c in frame.columns), None)
    speed_col = next((c for c in speed_candidates if c in frame.columns), None)

    if dist_col is not None:
        dist = pd.to_numeric(frame[dist_col], errors="coerce").clip(lower=0)
        frame["engineered_log_distance"] = np.log1p(dist)
        if speed_col is not None:
            speed = pd.to_numeric(frame[speed_col], errors="coerce")
            frame["engineered_speed_distance_ratio"] = speed / (dist + 1.0)
    return frame

X_all = add_domain_features(X_all)
X_test_all = add_domain_features(X_test_all)
X_test_all = X_test_all.reindex(columns=X_all.columns)

# Replace infinite values created by ratios with missing values; the imputer handles them.
X_all = X_all.replace([np.inf, -np.inf], np.nan)
X_test_all = X_test_all.replace([np.inf, -np.inf], np.nan)

print("Number of numeric features:", X_all.shape[1])
print("Feature names:")
print(X_all.columns.tolist())

Number of numeric features: 36
Feature names:
['num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first', 'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h', 'radial_growth_m', 'radial_growth_rate_m_per_h', 'centroid_displacement_m', 'centroid_speed_m_per_h', 'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos', 'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h', 'dist_slope_ci_0_5h', 'closing_speed_m_per_h', 'closing_speed_abs_m_per_h', 'projected_advance_m', 'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos', 'alignment_abs', 'cross_track_component', 'along_track_speed', 'event_start_hour', 'event_start_dayofweek', 'event_start_month', 'engineered_log_distance', 'engineered_speed_distance_ratio']


## 5. Choose models

We use gradient-boosted trees where available. For each horizon, the model predicts a probability directly. We do **not** use an arbitrary time-to-probability conversion.

If a library is missing, the notebook falls back to a scikit-learn histogram gradient boosting classifier. Install packages in the same Python environment as this notebook if you want to use XGBoost and LightGBM.

In [25]:
try:
    from xgboost import XGBClassifier
    HAVE_XGBOOST = True
except ImportError:
    HAVE_XGBOOST = False

try:
    from lightgbm import LGBMClassifier
    HAVE_LIGHTGBM = True
except ImportError:
    HAVE_LIGHTGBM = False

print("XGBoost available:", HAVE_XGBOOST)
print("LightGBM available:", HAVE_LIGHTGBM)

def make_model(model_name="xgb"):
    if model_name == "xgb" and HAVE_XGBOOST:
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                n_estimators=350,
                max_depth=4,
                learning_rate=0.035,
                subsample=0.85,
                colsample_bytree=0.85,
                min_child_weight=3,
                reg_lambda=2.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])
    if model_name == "lgb" and HAVE_LIGHTGBM:
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMClassifier(
                n_estimators=350,
                learning_rate=0.035,
                num_leaves=15,
                max_depth=-1,
                min_child_samples=20,
                reg_lambda=2.0,
                verbosity=-1,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=RANDOM_STATE
        ))
    ])

XGBoost available: True
LightGBM available: True


## 6. Build the horizon-specific targets

The labels are created separately for 12, 24, 48, and 72 hours. Rows with unknown status at a horizon are excluded from that horizon's training set.

In [26]:
def make_horizon_target(df, horizon):
    time = pd.to_numeric(df[TIME_COL], errors="coerce")
    event = pd.to_numeric(df[EVENT_COL], errors="coerce")

    # Per metadata: event=0 means no hit within the 72h window,
    # so at the 72h boundary every non-event is a known negative.
    WINDOW = 72
    known = ((event == 1) & time.notna()) | (
        (event == 0) & ((time >= horizon) | (horizon >= WINDOW))
    )
    hit_by_horizon = (event == 1) & (time <= horizon)

    y = hit_by_horizon.loc[known].astype(int)
    return known, y

for h in HORIZONS:
    known, y_h = make_horizon_target(train, h)
    print(f"{h}h: usable rows={int(known.sum())}, positives={int(y_h.sum())}, "
          f"positive rate={y_h.mean():.3f}" if len(y_h) else f"{h}h: no usable rows")

12h: usable rows=215, positives=49, positive rate=0.228
24h: usable rows=196, positives=63, positive rate=0.321
48h: usable rows=166, positives=66, positive rate=0.398
72h: usable rows=221, positives=69, positive rate=0.312


## 7. Validation split and baseline evaluation

We hold out part of the training data for a first diagnostic. This is a simple random split, not a guarantee of ideal generalization; if events are clustered by incident, time, or geography, a grouped split may be more appropriate.

We report Brier score (lower is better) and ROC AUC when both classes appear in the validation subset. The competition uses its own censor-aware scoring implementation, so these local diagnostics are useful but are not guaranteed to match the leaderboard score exactly.

In [27]:
all_idx = np.arange(len(train))
train_idx, val_idx = train_test_split(
    all_idx, test_size=0.20, random_state=RANDOM_STATE
)

validation_results = []
validation_predictions = {}

for horizon in HORIZONS:
    known, y_h = make_horizon_target(train, horizon)
    known_idx = np.flatnonzero(known.to_numpy())

    # Only use known observations for this horizon.
    fit_idx = np.intersect1d(train_idx, known_idx)
    eval_idx = np.intersect1d(val_idx, known_idx)

    y_fit = ((train.loc[fit_idx, EVENT_COL] == 1) &
             (train.loc[fit_idx, TIME_COL] <= horizon)).astype(int)
    y_eval = ((train.loc[eval_idx, EVENT_COL] == 1) &
              (train.loc[eval_idx, TIME_COL] <= horizon)).astype(int)

    if len(fit_idx) < 10 or len(eval_idx) < 5 or y_fit.nunique() < 2:
        print(f"Skipping {horizon}h validation: too few rows or only one training class.")
        continue

    model_xgb = make_model("xgb")
    model_xgb.fit(X_all.iloc[fit_idx], y_fit)
    p_xgb = model_xgb.predict_proba(X_all.iloc[eval_idx])[:, 1]

    if HAVE_LIGHTGBM:
        model_lgb = make_model("lgb")
        model_lgb.fit(X_all.iloc[fit_idx], y_fit)
        p_lgb = model_lgb.predict_proba(X_all.iloc[eval_idx])[:, 1]
        p_eval = 0.5 * p_xgb + 0.5 * p_lgb
    else:
        p_eval = p_xgb

    p_eval = np.clip(p_eval, 0, 1)
    brier = brier_score_loss(y_eval, p_eval)
    auc = roc_auc_score(y_eval, p_eval) if y_eval.nunique() == 2 else np.nan

    validation_results.append({
        "horizon_hours": horizon,
        "validation_rows": len(eval_idx),
        "positive_rate": y_eval.mean(),
        "brier_score_lower_is_better": brier,
        "roc_auc_higher_is_better": auc
    })
    validation_predictions[horizon] = (eval_idx, y_eval.to_numpy(), p_eval)

display(pd.DataFrame(validation_results))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,horizon_hours,validation_rows,positive_rate,brier_score_lower_is_better,roc_auc_higher_is_better
0,12,45,0.266667,0.072680,0.967172
1,24,39,0.410256,0.038878,0.970109
2,48,34,0.500000,0.014881,1.000000
3,72,45,0.400000,0.000743,1.000000


## 8. Train final models and predict the test set

For each horizon, we fit on every training row whose outcome is known at that horizon. We ensemble XGBoost and LightGBM probabilities equally when both are available. The weights are a baseline, not a proven optimum; change them only after comparing validation results.

In [28]:
test_probabilities = {}
final_models = {}

for horizon in HORIZONS:
    known, _ = make_horizon_target(train, horizon)
    fit_idx = np.flatnonzero(known.to_numpy())

    y_fit = ((train.loc[fit_idx, EVENT_COL] == 1) &
             (train.loc[fit_idx, TIME_COL] <= horizon)).astype(int)

    if len(fit_idx) == 0 or y_fit.nunique() < 2:
        # A constant fallback based on the usable training prevalence.
        fallback = float(y_fit.mean()) if len(y_fit) else float(train[EVENT_COL].mean())
        test_probabilities[horizon] = np.full(len(test), np.clip(fallback, 0, 1))
        print(f"{horizon}h: using constant fallback probability.")
        continue

    model_xgb = make_model("xgb")
    model_xgb.fit(X_all.iloc[fit_idx], y_fit)
    p_xgb = model_xgb.predict_proba(X_test_all)[:, 1]

    if HAVE_LIGHTGBM:
        model_lgb = make_model("lgb")
        model_lgb.fit(X_all.iloc[fit_idx], y_fit)
        p_lgb = model_lgb.predict_proba(X_test_all)[:, 1]
        p_final = 0.5 * p_xgb + 0.5 * p_lgb
        final_models[horizon] = {"xgb": model_xgb, "lgb": model_lgb}
    else:
        p_final = p_xgb
        final_models[horizon] = {"xgb": model_xgb}

    test_probabilities[horizon] = np.clip(p_final, 0, 1)
    print(f"{horizon}h: fitted on {len(fit_idx)} known observations.")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


12h: fitted on 215 known observations.
24h: fitted on 196 known observations.
48h: fitted on 166 known observations.
72h: fitted on 221 known observations.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 9. Enforce monotonic probabilities

The probability of a hit by 12 hours should not exceed the probability of a hit by 24, 48, or 72 hours. Small model inconsistencies can violate this, so we apply a row-wise cumulative maximum.

In [29]:
prob_matrix = np.column_stack([test_probabilities[h] for h in HORIZONS])
prob_matrix = np.maximum.accumulate(prob_matrix, axis=1)

submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    "prob_12h": prob_matrix[:, 0],
    "prob_24h": prob_matrix[:, 1],
    "prob_48h": prob_matrix[:, 2],
    "prob_72h": prob_matrix[:, 3],
})

display(submission.head())
display(submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].describe())

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.007947,0.008248,0.011284,0.012109
1,13353600,0.560489,0.981678,0.981678,0.981678
2,13942327,0.006585,0.007500,0.011284,0.012109
3,16112781,0.667572,0.906972,0.969342,0.973944
4,17132808,0.027000,0.027000,0.027000,0.027000


,prob_12h,prob_24h,prob_48h,prob_72h
count,95.000000,95.000000,95.000000,95.000000
mean,0.175685,0.276373,0.289591,0.297617
std,0.292331,0.411435,0.426992,0.438259
min,0.003312,0.007201,0.010357,0.012109
25%,0.006413,0.010288,0.013261,0.013261
50%,0.010615,0.013325,0.015122,0.015122
75%,0.313524,0.823937,0.907035,0.973843
max,0.971075,0.981678,0.981678,0.981678


## 10. Validate submission requirements

Checks:
- required columns and order;
- IDs match the test set exactly, with no duplicates;
- probabilities are finite and within [0, 1];
- probabilities are monotonic across horizons.

In [30]:
expected_columns = [ID_COL, "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
submission = submission[expected_columns]

assert submission[ID_COL].equals(test[ID_COL].reset_index(drop=True)), "Submission IDs/order do not match test."
assert submission[ID_COL].is_unique, "Duplicate IDs in submission."
prob_cols = ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]
assert np.isfinite(submission[prob_cols].to_numpy()).all(), "Non-finite probabilities found."
assert ((submission[prob_cols] >= 0) & (submission[prob_cols] <= 1)).all().all(), "Probabilities outside [0, 1]."
assert (submission["prob_12h"] <= submission["prob_24h"] + 1e-12).all()
assert (submission["prob_24h"] <= submission["prob_48h"] + 1e-12).all()
assert (submission["prob_48h"] <= submission["prob_72h"] + 1e-12).all()

print("All submission checks passed.")

All submission checks passed.


## 11. Save the Kaggle submission

The file will be saved in the notebook's current working directory. Upload `submission_wildfire_baseline.csv` to the competition submission page.

In [31]:
output_path = DATA_DIR / "submission_wildfire_baseline.csv"
submission.to_csv(output_path, index=False)
print("Saved:", output_path.resolve())

Saved: /content/submission_wildfire_baseline.csv


## 12. Important next steps before claiming a strong result

1. Submit this baseline and record the public score.
2. Compare changes using a validation strategy that matches the competition's censor-aware metric as closely as possible.
3. Check whether rows are grouped by fire incident or location; if so, use grouped validation to reduce leakage.
4. Inspect feature definitions in the metadata. Do not remove columns just because one fitted model gives them low feature importance.
5. Tune ensemble weights and model parameters based on validation results, not because a probability distribution looks visually balanced.
6. Do not interpret a good-looking mean probability as proof of calibration. Calibration must be checked against known outcomes using a valid evaluation procedure.

This notebook is a reproducible starting point, not a guarantee of a top leaderboard position.